# 02 — Time Travel Demo
Demonstrate Delta Lake time-travel: query past versions, audit changes, restore.

In [ ]:
import sys; sys.path.insert(0, '..')
from src.utils.spark_session import get_spark
spark = get_spark('../config/env.yaml')

import yaml
with open('../config/env.yaml') as f:
    cfg = yaml.safe_load(f)

silver_customers = cfg['layers']['silver'] + '/customers'
print('Silver customers path:', silver_customers)

In [ ]:
# Show version history
spark.sql(f"DESCRIBE HISTORY delta.`{silver_customers}`").show(truncate=False)

In [ ]:
# Query at version 0 (right after initial load)
spark.read.format('delta').option('versionAsOf', 0).load(silver_customers).show(5)

In [ ]:
# Compare current vs version 0 — find new/changed customers
from pyspark.sql import functions as F

current = spark.read.format('delta').load(silver_customers).filter('_is_current = true')
v0      = spark.read.format('delta').option('versionAsOf', 0).load(silver_customers)

new_customers = current.join(v0, 'customer_id', 'left_anti')
print(f'New customers since v0: {new_customers.count():,}')
new_customers.show(5)